In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("datasets/processed/gp_weather.csv")
df = df.dropna(subset=["weather_raw"]).copy()
df

,raceId,year,url_gp,weather_raw
0,839,1950,http://en.wikipedia.org/wiki/1950_Italian_Gran...,"Warm, hot and sunny"
1,838,1950,http://en.wikipedia.org/wiki/1950_French_Grand...,Hot and sunny
2,837,1950,http://en.wikipedia.org/wiki/1950_Belgian_Gran...,"Warm, dry and sunny"
3,836,1950,http://en.wikipedia.org/wiki/1950_Swiss_Grand_...,"Warm, dry and sunny"
6,833,1950,http://en.wikipedia.org/wiki/1950_British_Gran...,"Sunny, mild, dry."
...,...,...,...,...
1120,1123,2024,https://en.wikipedia.org/wiki/2024_Australian_...,Sunny
1121,1122,2024,https://en.wikipedia.org/wiki/2024_Saudi_Arabi...,Clear
1122,1121,2024,https://en.wikipedia.org/wiki/2024_Bahrain_Gra...,Clear
1123,1131,2024,https://en.wikipedia.org/wiki/2024_Austrian_Gr...,Partly cloudy


In [2]:
from sentence_transformers import SentenceTransformer, util

In [3]:
weather = {
    'Dry' : ['sunny', 'mild', 'dry', 'overcast', 'hot', 'clear'],
    'Wet' : ['rain', 'wet', 'light rain', 'heavy rain', 'rainy'],
    'Variable' : ['half-distance', 'start', 'finish', 'part', 'then' , 'rain', 'wet', 'dry', 'sunny','hot', 'end'] 
}

In [4]:
weather_names = list(weather.keys())
category_texts = [" ".join(keywords) for keywords in weather.values()]

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [6]:
category_embeddings = model.encode(category_texts, normalize_embeddings=True)

In [9]:
def find_weather(user_text: str):
    if not isinstance(user_text, str) or user_text.strip() == '':
        return 'Unknown'
    user_embedding = model.encode(user_text, normalize_embeddings=True)
    scores = util.cos_sim(user_embedding, category_embeddings)[0]
    top_result_idx = scores.argmax().item()
    return weather_names[top_result_idx]


In [10]:
print(find_weather("Fine and Dry, showers threatening at end"))

Dry


In [11]:
df['weather'] = df['weather_raw'].apply(find_weather)


In [17]:
weather_history = df[['raceId', 'weather']]
weather_history

,raceId,weather
0,839,Dry
1,838,Dry
2,837,Dry
3,836,Dry
6,833,Dry
...,...,...
1120,1123,Dry
1121,1122,Dry
1122,1121,Dry
1123,1131,Dry


In [20]:
weather_history.to_csv("C:/Users/danil/Desktop/PROJECTS/formula1/datasets/processed/weather_conditions.csv", index=False)